# آموزش آزمایشی کنترلر پرواز با PPO در Google Colab

این نوت‌بوک هستهٔ فیزیک C++ خود SkySim را از همان revision و با همان patch تنظیمات پروژه که در ساخت نسخهٔ مرورگری استفاده می‌شود، به‌صورت native کامپایل می‌کند؛ سپس Gymnasium و PPO را مستقیماً روی آن اجرا می‌کند. این مسیر دیگر به سرور Godot/WebSocket نیاز ندارد.

**محدودیت مهم:** منبع C++ مشترک است، اما برابری عددی نسخهٔ native با WASM هنوز باید با تست‌های رگرسیون تأیید شود. این آموزش، شبیه‌سازی معتبرِ کوادکوپتر واقعی یا تأیید ایمنی پرواز نیست. مقادیر فعلی `config/drone.json` همچنان placeholder هستند و این نوت‌بوک آن‌ها را تغییر نمی‌دهد.


In [ ]:
!git clone https://github.com/samanramezani1377-hub/3d-fly-test.git
%cd /content/3d-fly-test
!python -m pip install -r training/requirements.txt


## ساخت هستهٔ فیزیک native

این مرحله revision ثابت SkySim را دریافت می‌کند، patch یکسان تنظیمات پروژه و مدل باتری را اعمال می‌کند و فایل shared library را با `g++` می‌سازد. اگر ساخت ناموفق شد، آموزش را ادامه نده.

In [ ]:
!python training/build_native_core.py


## اجرای آموزش PPO

ابتدا یک اجرای کوتاه برای اطمینان از سالم بودن اتصال Gymnasium انجام بده. برای اجرای طولانی‌تر مقدار `TIMESTEPS` را بیشتر کن. Colab ممکن است جلسه را متوقف کند؛ نتایج را در پایان دانلود کن.

In [ ]:
TIMESTEPS = 5000  # smoke test؛ بعداً می‌توانی 200_000 یا بیشتر بگذاری
RUN_NAME = 'colab-prototype-hover'
!python training/train_ppo.py --timesteps {TIMESTEPS} --seed 42 --run-name {RUN_NAME}


In [ ]:
!python training/evaluate.py --model training/runs/{RUN_NAME}/final_model.zip --episodes 10 --output training/runs/{RUN_NAME}/evaluation.json
!cd training/runs && zip -r /content/{RUN_NAME}-results.zip {RUN_NAME}
from google.colab import files
files.download(f'/content/{RUN_NAME}-results.zip')


## یادآوری برای مرحلهٔ بعد

بعد از آماده‌شدن مدل فیزیکی واقعی، باید پارامترهای واقعی از اندازه‌گیری و اطلاعات قطعات وارد شوند. پیش از اتکا به آموزش، تست‌های رگرسیون مشترک native/WASM باید اجرا شوند؛ سپس می‌توان بررسی کرد که آیا لازم است دامنهٔ تصادفی‌سازی و تابع پاداش تغییر کند. این مدل فعلاً برای پرواز واقعی استفاده نشود.